# 07_incident_clustering

This notebook performs unsupervised clustering on ASRS narratives to identify
operational themes, human-factor patterns, and recurring safety issues.

We use TF-IDF vectorization + KMeans clustering for interpretable, offline clustering.

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [3]:
df = pd.read_csv("../data/processed/asrs_risk_scored_3yrs.csv")

print("Loaded:", df.shape)
df["Narrative"].head()

Loaded: (16535, 137)


C:\Users\jenny\AppData\Local\Temp\ipykernel_19180\1805580239.py:1: DtypeWarning: Columns (7,8,15,19,20,38,39,40,41,42,43,44,45,46,47,48,49,50,59,63,78,79,81,82,83,86,89,99,100,110,111,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/asrs_risk_scored_3yrs.csv")


0    To be clear I did not have a lot going on and ...
1    While descending into PIB out of 14;000 ft. MS...
2    I was boarding the aircraft when a my destinat...
3    I had a ferry flight that went out of service ...
4    We were flying at 2500 ft. I started losing el...
Name: Narrative, dtype: object

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = text.replace("\n", " ")
    return text

df["clean_narrative"] = df["Narrative"].apply(clean_text)

In [5]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df["clean_narrative"])
X.shape

(16535, 5000)

In [6]:
scores = {}
for k in [5, 7, 10, 12]:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    scores[k] = score

scores

{5: 0.011079060103458093,
 7: 0.011585041392951028,
 10: 0.01421237921380049,
 12: 0.014651016792805041}

In [7]:
BEST_K = max(scores, key=scores.get)
print("Using K =", BEST_K)

kmeans = KMeans(n_clusters=BEST_K, random_state=42)
df["cluster"] = kmeans.fit_predict(X)

Using K = 12


In [ ]:
terms = vectorizer.get_feature_names_out()
centers = kmeans.cluster_centers_

def top_terms(cluster_idx, n=15):
    center = centers[cluster_idx]
    top_ids = center.argsort()[::-1][:n]
    return [terms[i] for i in top_ids]

cluster_terms = {i: top_terms(i) for i in range(BEST_K)}
cluster_terms

In [9]:
df["cluster_terms"] = df["cluster"].apply(lambda c: ", ".join(cluster_terms[c]))
df[["cluster", "cluster_terms"]].head()

,cluster,cluster_terms
0,5,"aircraft, traffic, airspace, ft, controller, s..."
1,1,"aircraft, flight, maintenance, zzz, crew, land..."
2,1,"aircraft, flight, maintenance, zzz, crew, land..."
3,1,"aircraft, flight, maintenance, zzz, crew, land..."
4,1,"aircraft, flight, maintenance, zzz, crew, land..."


In [10]:
output_file = "../data/processed/asrs_clustered_3yrs.csv"
df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Final shape:", df.shape)

Saved: ../data/processed/asrs_clustered_3yrs.csv
Final shape: (16535, 140)
